# 面试问题：MinHash 与 LSH 如何做近重复检测，band 参数怎样影响召回？

可以直接复述的回答是：先把文档转成 shingles 集合，Jaccard 是近重复的精确相似度。MinHash 用多组可复现哈希记录每组最小值，两份集合签名相等比例是 Jaccard 的无偏估计。LSH 再把签名切成 bands，只有某个 band 完全相同的文档才成为候选。band 少而每 band 行数多时过滤严格、精度高但容易漏召回；band 多且行数少时召回更高但候选更多。候选之后仍必须计算精确 Jaccard 才能最终判重。下面手写 24 维签名和两种 band 配置，并打印真实碰撞桶。

## 真实案例：十篇资讯的转载与改写去重

文本预先用空格表示词边界，包含轨道交通、暴雨预警和手机发布三组近重复，以及三篇独立内容。数据为教学构造的离线快照，不代表生产版权判定。

In [1]:
import hashlib  # 导入稳定摘要用于跨进程一致 token ID
import itertools  # 导入文档对组合生成器
import random  # 导入确定性 MinHash 参数生成器
documents = [  # 定义十篇带主题的资讯文本
    ("N-01", "交通局 宣布 东环线 明年 三月 试运行 许可 复核 完成"),  # 轨道官方原文
    ("N-02", "交通局 今日 宣布 东环线 明年 三月 开始 试运行 许可 复核 完成"),  # 轨道轻度转载
    ("N-03", "东环线 明年 三月 试运行 交通局 宣布 许可 复核 已经 完成"),  # 轨道改写稿
    ("N-04", "气象台 发布 暴雨 红色 预警 城区 学校 今日 停课"),  # 暴雨原文
    ("N-05", "气象台 今日 发布 暴雨 红色 预警 城区 学校 停课"),  # 暴雨近重复
    ("N-06", "品牌 发布 新款 手机 支持 卫星 通信 与 快速 充电"),  # 手机发布原文
    ("N-07", "品牌 今日 发布 新款 手机 支持 卫星 通信 和 快速 充电"),  # 手机发布近重复
    ("N-08", "本周 联赛 决赛 主队 加时 获胜 球迷 庆祝"),  # 独立体育资讯
    ("N-09", "咖啡 生豆 价格 回落 门店 推出 春季 新品"),  # 独立消费资讯
    ("N-10", "开源 数据库 发布 新版本 改进 查询 优化器"),  # 独立技术资讯
]  # 结束十篇资讯
def shingles(text):  # 把预分词文本转换成一词 shingle 集合
    return set(text.split())  # 用集合去重并保留 Jaccard 语义
shingle_sets = {doc_id: shingles(text) for doc_id, text in documents}  # 建立文档到 shingle 集合映射
print("输入预览：doc | shingle_count | shingles")  # 输出文档集合表头
for doc_id, text in documents:  # 逐条展示十篇资讯
    print(f"{doc_id} | {len(shingle_sets[doc_id]):2d} | {sorted(shingle_sets[doc_id])}")  # 展示真实比较单元
print("全部文档对数量：", len(list(itertools.combinations(shingle_sets, 2))))  # 展示精确基线比较成本

输入预览：doc | shingle_count | shingles
N-01 |  9 | ['三月', '东环线', '交通局', '复核', '完成', '宣布', '明年', '许可', '试运行']
N-02 | 11 | ['三月', '东环线', '交通局', '今日', '复核', '完成', '宣布', '开始', '明年', '许可', '试运行']
N-03 | 10 | ['三月', '东环线', '交通局', '复核', '完成', '宣布', '已经', '明年', '许可', '试运行']
N-04 |  9 | ['今日', '停课', '发布', '城区', '学校', '暴雨', '气象台', '红色', '预警']
N-05 |  9 | ['今日', '停课', '发布', '城区', '学校', '暴雨', '气象台', '红色', '预警']
N-06 | 10 | ['与', '充电', '卫星', '发布', '品牌', '快速', '手机', '支持', '新款', '通信']
N-07 | 11 | ['今日', '充电', '卫星', '发布', '和', '品牌', '快速', '手机', '支持', '新款', '通信']
N-08 |  8 | ['主队', '决赛', '加时', '庆祝', '本周', '球迷', '联赛', '获胜']
N-09 |  8 | ['价格', '咖啡', '回落', '推出', '新品', '春季', '生豆', '门店']
N-10 |  7 | ['优化器', '发布', '开源', '改进', '数据库', '新版本', '查询']
全部文档对数量： 45


## Baseline / 基线：所有文档对计算精确 Jaccard

十篇文档共有 45 对。阈值设为 0.55，精确结果既是最终判重规则，也是 LSH candidate recall 的权威 gold。

In [2]:
def jaccard(left, right):  # 手写两个 shingle 集合的精确 Jaccard
    union = left | right  # 计算并集用于归一化
    return len(left & right) / len(union) if union else 1.0  # 返回交集占并集比例
all_pairs = list(itertools.combinations(shingle_sets.keys(), 2))  # 枚举十文档全部四十五对
exact_similarities = {pair: jaccard(shingle_sets[pair[0]], shingle_sets[pair[1]]) for pair in all_pairs}  # 计算每对精确相似度
gold_pairs = {pair for pair, similarity in exact_similarities.items() if similarity >= 0.55}  # 按阈值形成近重复 gold
print("精确近重复：pair | Jaccard")  # 输出权威近重复表头
for pair in sorted(gold_pairs):  # 遍历所有达到阈值的文档对
    print(f"{pair} | {exact_similarities[pair]:.3f}")  # 展示真实集合相似度
print(f"全对比较={len(all_pairs)}，gold_pairs={len(gold_pairs)}")  # 汇总基线计算成本

精确近重复：pair | Jaccard
('N-01', 'N-02') | 0.818
('N-01', 'N-03') | 0.900
('N-02', 'N-03') | 0.750
('N-04', 'N-05') | 1.000
('N-06', 'N-07') | 0.750
全对比较=45，gold_pairs=5


## 核心实现：24 维 MinHash 签名

每个词先用 SHA-1 得到稳定整数，再使用 `h(x)=(a*x+b) mod p` 的 24 组参数。输出 N-01/N-02 前八维签名及相等比例。

In [3]:
prime = 4294967311  # 选择大于三十二位空间的素数模数
random_generator = random.Random(1705)  # 创建固定 MinHash 参数随机源
hash_parameters = [(random_generator.randrange(1, prime), random_generator.randrange(0, prime)) for _ in range(24)]  # 生成二十四组线性哈希参数
def stable_token_id(token):  # 把中文词项映射为跨进程稳定整数
    digest = hashlib.sha1(token.encode("utf-8")).digest()[:4]  # 截取 SHA-1 前四字节
    return int.from_bytes(digest, "big")  # 按大端序转换为无符号整数
def minhash_signature(token_set):  # 手写一个集合的二十四维 MinHash 签名
    token_ids = [stable_token_id(token) for token in token_set]  # 预计算集合内稳定整数 ID
    signature = []  # 收集每组哈希的最小值
    for multiplier, offset in hash_parameters:  # 遍历二十四组伪随机排列
        minimum = min((multiplier * token_id + offset) % prime for token_id in token_ids)  # 取当前排列下集合最小哈希
        signature.append(minimum)  # 追加当前 MinHash 分量
    return tuple(signature)  # 返回可哈希的固定长度签名
signatures = {doc_id: minhash_signature(token_set) for doc_id, token_set in shingle_sets.items()}  # 为十篇文档计算签名
def estimated_jaccard(left_id, right_id):  # 用签名相等比例估计 Jaccard
    matches = sum(left == right for left, right in zip(signatures[left_id], signatures[right_id]))  # 统计二十四维相等位置
    return matches / len(hash_parameters)  # 返回 MinHash 相似度估计
print("N-01 signature[:8]：", signatures["N-01"][:8])  # 展示原文签名前八维
print("N-02 signature[:8]：", signatures["N-02"][:8])  # 展示转载签名前八维
print(f"N-01/N-02 exact={exact_similarities[('N-01', 'N-02')]:.3f}，MinHash estimate={estimated_jaccard('N-01', 'N-02'):.3f}")  # 对比精确与估计相似度

N-01 signature[:8]： (48488502, 66629907, 246214311, 18023577, 81134980, 310535450, 1088569828, 418049009)
N-02 signature[:8]： (48488502, 66629907, 246214311, 18023577, 81134980, 210901487, 803791274, 418049009)
N-01/N-02 exact=0.818，MinHash estimate=0.833


## LSH 分桶：碰撞才进入精确复核

`bands * rows = 24`。函数打印拥有两个以上文档的碰撞桶，并返回去重候选对。

In [4]:
def lsh_candidates(bands, rows):  # 手写签名 band 分桶和候选生成
    buckets = {}  # 建立 band 与签名片段到文档列表映射
    for doc_id, signature in signatures.items():  # 遍历十篇文档签名
        for band in range(bands):  # 逐个切分当前签名 band
            start = band * rows  # 计算 band 起始维度
            key = (band, signature[start:start + rows])  # 用 band 编号和完整片段组成桶键
            buckets.setdefault(key, []).append(doc_id)  # 将文档加入对应 LSH 桶
    candidates = set()  # 收集任一 band 碰撞的文档对
    collisions = []  # 保存多人桶用于过程展示
    for key, doc_ids in buckets.items():  # 遍历全部分桶结果
        if len(doc_ids) >= 2:  # 只处理至少两个文档的碰撞桶
            collisions.append((key[0], doc_ids.copy()))  # 保存 band 编号和碰撞文档
            for left, right in itertools.combinations(sorted(doc_ids), 2):  # 枚举桶内所有候选对
                candidates.add((left, right))  # 用集合跨 band 去重候选
    return candidates, collisions  # 返回候选集合与可解释碰撞桶
fixed_candidates, fixed_collisions = lsh_candidates(12, 2)  # 使用多 band 少行数的召回配置
print("12 bands × 2 rows 碰撞桶：band | docs")  # 输出 LSH 中间过程表头
for band, doc_ids in fixed_collisions:  # 遍历真实多人桶
    print(f"{band:2d} | {doc_ids}")  # 展示哪些文档因签名片段相同成为候选
print("去重候选对：", sorted(fixed_candidates))  # 展示进入精确 Jaccard 复核的文档对

12 bands × 2 rows 碰撞桶：band | docs
 0 | ['N-01', 'N-02', 'N-03']
 1 | ['N-01', 'N-02', 'N-03']
 2 | ['N-01', 'N-03']
 3 | ['N-01', 'N-03']
 4 | ['N-01', 'N-02', 'N-03']
 5 | ['N-01', 'N-02', 'N-03']
 6 | ['N-01', 'N-03']
 7 | ['N-01', 'N-03']
 8 | ['N-01', 'N-02', 'N-03']
 9 | ['N-01', 'N-02', 'N-03']
10 | ['N-01', 'N-02', 'N-03']
11 | ['N-01', 'N-02', 'N-03']
 0 | ['N-04', 'N-05']
 1 | ['N-04', 'N-05']
 2 | ['N-04', 'N-05']
 3 | ['N-04', 'N-05']
 4 | ['N-04', 'N-05', 'N-06', 'N-07']
 5 | ['N-04', 'N-05']
 6 | ['N-04', 'N-05']
 7 | ['N-04', 'N-05']
 8 | ['N-04', 'N-05']
 9 | ['N-04', 'N-05']
10 | ['N-04', 'N-05']
11 | ['N-04', 'N-05']
 1 | ['N-06', 'N-07']
 2 | ['N-06', 'N-07']
 3 | ['N-06', 'N-07']
 8 | ['N-06', 'N-07']
10 | ['N-06', 'N-07']
11 | ['N-06', 'N-07']
去重候选对： [('N-01', 'N-02'), ('N-01', 'N-03'), ('N-02', 'N-03'), ('N-04', 'N-05'), ('N-04', 'N-06'), ('N-04', 'N-07'), ('N-05', 'N-06'), ('N-05', 'N-07'), ('N-06', 'N-07')]


## 失败案例与修正、候选质量结果

严格配置 `2 bands × 12 rows` 要求连续十二维完全相等，遗漏部分 gold。修正为 `12 × 2` 后提高候选召回，再用精确 Jaccard 阈值过滤假阳性。

In [5]:
strict_candidates, strict_collisions = lsh_candidates(2, 12)  # 复现少 band 多行数的严格配置
strict_missed = gold_pairs - strict_candidates  # 计算严格 LSH 漏掉的真实近重复
fixed_missed = gold_pairs - fixed_candidates  # 计算高召回配置遗漏集合
verified_pairs = {pair for pair in fixed_candidates if exact_similarities[pair] >= 0.55}  # 对候选执行精确 Jaccard 最终判重
strict_recall = len(gold_pairs & strict_candidates) / len(gold_pairs)  # 计算严格配置候选召回率
fixed_recall = len(gold_pairs & fixed_candidates) / len(gold_pairs)  # 计算修正配置候选召回率
fixed_precision = len(gold_pairs & fixed_candidates) / len(fixed_candidates) if fixed_candidates else 0.0  # 计算修正候选精度
print("pair | exact_J | minhash_est | strict_candidate | fixed_candidate | verified")  # 输出逐 gold 对结果表头
for pair in sorted(gold_pairs):  # 遍历全部权威近重复
    estimate = estimated_jaccard(*pair)  # 计算当前对签名相等比例
    print(f"{pair} | {exact_similarities[pair]:.3f} | {estimate:.3f} | {pair in strict_candidates} | {pair in fixed_candidates} | {pair in verified_pairs}")  # 展示 band 参数对召回的影响
print("严格配置漏召回：", sorted(strict_missed))  # 展示真实失败样本
print("修正配置漏召回：", sorted(fixed_missed))  # 展示提高召回后的结果
print(f"候选数：strict={len(strict_candidates)}，fixed={len(fixed_candidates)}，全对={len(all_pairs)}")  # 对比近似候选成本
print(f"recall：strict={strict_recall:.1%}，fixed={fixed_recall:.1%}，fixed candidate precision={fixed_precision:.1%}")  # 汇总候选质量

pair | exact_J | minhash_est | strict_candidate | fixed_candidate | verified
('N-01', 'N-02') | 0.818 | 0.833 | False | True | True
('N-01', 'N-03') | 0.900 | 1.000 | True | True | True
('N-02', 'N-03') | 0.750 | 0.833 | False | True | True
('N-04', 'N-05') | 1.000 | 1.000 | True | True | True
('N-06', 'N-07') | 0.750 | 0.708 | False | True | True
严格配置漏召回： [('N-01', 'N-02'), ('N-02', 'N-03'), ('N-06', 'N-07')]
修正配置漏召回： []
候选数：strict=2，fixed=9，全对=45
recall：strict=40.0%，fixed=100.0%，fixed candidate precision=55.6%


## 结果解读

MinHash 估计来自签名位置相等比例，并不保证等于精确 Jaccard。LSH 的职责只是便宜地产生候选；修正配置提高召回的代价是更多碰撞，最终判重仍回到原 shingles 的精确相似度。

## 生产边界

教学实验仅十篇短文和一词 shingles。生产去重要做中文规范化、模板噪声移除、标题/正文加权、跨语言和时间窗口分桶，还要持久化 hash seed 与签名版本。大规模 LSH 桶需限制热点，候选复核要支持增量删除和版权审计；阈值应按误杀与漏检成本标注评估。

## 最小回归测试

In [6]:
assert len(documents) >= 6 and len(gold_pairs) >= 3  # 保证案例包含多篇文档和多组近重复
assert all(len(signature) == 24 for signature in signatures.values())  # 保证每篇文档真实生成完整 MinHash 签名
assert len(strict_missed) > 0  # 保证严格 band 配置漏召回失败真实复现
assert fixed_recall > strict_recall  # 保证增加 band 后候选召回提高
assert fixed_missed == set()  # 保证修正配置覆盖全部教学 gold
assert verified_pairs == gold_pairs  # 保证精确 Jaccard 复核恢复权威判重集合
assert len(fixed_candidates) < len(all_pairs)  # 保证 LSH 候选少于全对比较